In [2]:
import os
import cv2
import json
import torch
import random
import numpy as np

from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.data.datasets import register_coco_instances
from detectron2.data import MetadataCatalog, DatasetCatalog
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader


In [ ]:
import os
DATASET_PATH = "/kaggle/input/firesmoke-detection"
import numpy as np

np.random.seed(42)

for root, dirs, files in os.walk(DATASET_PATH):
    level = root.replace(DATASET_PATH, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for f in files[:5]:
        print(f"{subindent}{f}")

firesmoke-detection-yolo-v9/
  Fire and Smoke Dataset/
    data.yaml
    valid/
      labels/
        0071_JPG_jpg.rf.1504b5efdba534248c305c3044b2f308.txt
        Img_1737_jpg.rf.996238a678babda387e5be515085f75e.txt
        Img_1701_jpg.rf.971963cd8d87bd92856a31027ff21b8c.txt
        Img_232_jpg.rf.648657174159930fdf683d5afb4112dc.txt
        Img_1378_jpg.rf.8793ab2ca65c7d605aa5bd40996b7b9b.txt
      images/
        02575_jpg.rf.06efaf360de2df21695387a3b7379a76.jpg
        WEBFire1709_jpg.rf.9bacd7777197fe1ebd6651d393f85f50.jpg
        Img_574_jpg.rf.0474e0ce7ebd8279c32d3d11684d03d2.jpg
        000440_jpg.rf.d6d38e8acc97898a135c92ffcffb69f0.jpg
        Img_368_jpg.rf.1f85019248184f6988718635943e0644.jpg
    test/
      labels/
        Img_1213_jpg.rf.2ef4ba4811c39a1efb3e5e815387a63b.txt
        Img_1579_jpg.rf.b713d8ff00ce9353c1492bbfd157227c.txt
        Img_1814_jpg.rf.259ac6021ee2f1f72419327236187ba6.txt
        IMG_5398-1-_jpeg_jpg.rf.9e11e4b91ba43cfb6baecb041978db4c.txt
        fir

In [4]:
import os
import random
import shutil
from pathlib import Path

def create_subset(source_dir, dest_dir, fraction=0.2):
    """
    fraction: 0.2 means 20% of the data
    """
    images_path = Path(source_dir) / 'images'
    labels_path = Path(source_dir) / 'labels'
    
    # Create new directories
    os.makedirs(Path(dest_dir) / 'images', exist_ok=True)
    os.makedirs(Path(dest_dir) / 'labels', exist_ok=True)

    files = [f for f in os.listdir(images_path) if f.endswith(('.jpg', '.jpeg', '.png'))]
    subset_size = int(len(files) * fraction)
    subset_files = random.sample(files, subset_size)

    for file_name in subset_files:
        # Copy Image
        shutil.copy(images_path / file_name, Path(dest_dir) / 'images' / file_name)
        
        # Copy corresponding Label
        label_name = os.path.splitext(file_name)[0] + '.txt'
        if os.path.exists(labels_path / label_name):
            shutil.copy(labels_path / label_name, Path(dest_dir) / 'labels' / label_name)

    print(f"Moved {subset_size} images to {dest_dir}")

# Usage for Train and Val
create_subset('/kaggle/input/firesmoke-detection-yolo-v9/Fire and Smoke Dataset/train', '/kaggle/working/fire_subset/train', fraction=0.2)
# Usage for Train and Val
create_subset('/kaggle/input/firesmoke-detection-yolo-v9/Fire and Smoke Dataset/valid', '/kaggle/working/fire_subset/val', fraction=0.2)
create_subset('/kaggle/input/firesmoke-detection-yolo-v9/Fire and Smoke Dataset/test', '/kaggle/working/fire_subset/test', fraction=0.2)


import yaml

# Define your project paths (Update these to your actual subset folders)
dataset_config = {
    'train': '/kaggle/working/fire_subset/train/images',
    'val': '/kaggle/working/fire_subset/val/images',
    'test': '/kaggle/working/fire_subset/test',
    'nc': 2,
    'names': ['fire', 'smoke']
}

# Save the dictionary to a YAML file
yaml_path = '/kaggle/working/fire_data.yaml'

with open(yaml_path, 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

Moved 7139 images to /kaggle/working/fire_subset/train
Moved 978 images to /kaggle/working/fire_subset/val
Moved 451 images to /kaggle/working/fire_subset/test


# Preprocessing

In [5]:
import os, json, cv2

ROOT = "/kaggle/working/fire_subset"   # 🔥 20% subset
OUT  = "/kaggle/working/fire_coco"
os.makedirs(OUT, exist_ok=True)

CLASSES = ["fire", "smoke"]

def convert_split(split):
    images_dir = f"{ROOT}/{split}/images"
    labels_dir = f"{ROOT}/{split}/labels"

    coco = {
        "info": {
            "description": "Fire / Smoke Detection Dataset",
            "version": "1.0",
            "year": 2024,
            "contributor": "YOLO → COCO conversion",
            "date_created": "2024-01-01"
        },
        "images": [],
        "annotations": [],
        "categories": []
    }


    # Categories
    for i, cls_name in enumerate(CLASSES):
        coco["categories"].append({
            "id": i,
            "name": cls_name,
            "supercategory": "object"
        })

    ann_id = 0
    img_id = 0

    for img_name in sorted(os.listdir(images_dir)):
        if not img_name.lower().endswith((".jpg", ".png", ".jpeg")):
            continue

        img_path = os.path.join(images_dir, img_name)
        img = cv2.imread(img_path)

        if img is None:
            print(f"⚠️ Could not read image {img_name}, skipping")
            continue

        h, w = img.shape[:2]

        coco["images"].append({
            "id": img_id,
            "file_name": img_name,
            "width": w,
            "height": h
        })

        label_path = os.path.join(labels_dir, img_name.rsplit(".", 1)[0] + ".txt")
        if os.path.exists(label_path):
            with open(label_path) as f:
                for line in f:
                    parts = line.strip().split()

                    # Must have at least 5 values
                    if len(parts) < 5:
                        continue

                    # Take ONLY the first 5 (YOLO format)
                    cls, xc, yc, bw, bh = map(float, parts[:5])

                    x = (xc - bw / 2) * w
                    y = (yc - bh / 2) * h
                    bw = bw * w
                    bh = bh * h

                    coco["annotations"].append({
                        "id": ann_id,
                        "image_id": img_id,
                        "category_id": int(cls),
                        "bbox": [x, y, bw, bh],
                        "area": bw * bh,
                        "iscrowd": 0
                    })
                    ann_id += 1

        img_id += 1

    with open(f"{OUT}/{split}.json", "w") as f:
        json.dump(coco, f)

    print(f"✅ {split}: {len(coco['images'])} images converted")


# 🔁 Convert ONLY the subset
convert_split("train")

# ⚠️ Use ONE of these depending on your folder name
convert_split("val")


convert_split("test")


✅ train: 7139 images converted
✅ val: 978 images converted
✅ test: 451 images converted


In [6]:
import json

with open("/kaggle/working/fire_coco/train.json") as f:
    coco = json.load(f)

print("Images:", len(coco["images"]))
print("Annotations:", len(coco["annotations"]))
print("Categories:", coco["categories"])
print("First annotation:", coco["annotations"][0])


Images: 7139
Annotations: 7329
Categories: [{'id': 0, 'name': 'fire', 'supercategory': 'object'}, {'id': 1, 'name': 'smoke', 'supercategory': 'object'}]
First annotation: {'id': 0, 'image_id': 0, 'category_id': 1, 'bbox': [8.421053, 87.21568599999999, 28.502024, 199.529412], 'area': 5686.992089529888, 'iscrowd': 0}


In [7]:
from detectron2.data.datasets import register_coco_instances
from detectron2.data import MetadataCatalog

CLASSES = ["fire", "smoke"]

register_coco_instances(
    "fire_train",
    {},
    "/kaggle/working/fire_coco/train.json",
    "/kaggle/working/fire_subset/train/images"
)

register_coco_instances(
    "fire_val",
    {},
    "/kaggle/working/fire_coco/val.json",
    "/kaggle/working/fire_subset/val/images"
)

register_coco_instances(
    "fire_test",
    {},
    "/kaggle/working/fire_coco/test.json",
    "/kaggle/working/fire_subset/test/images"
)

MetadataCatalog.get("fire_train").thing_classes = CLASSES
MetadataCatalog.get("fire_val").thing_classes = CLASSES
MetadataCatalog.get("fire_test").thing_classes = CLASSES


In [8]:
import json

for split in ["train", "val", "test"]:
    path = f"/kaggle/working/fire_coco/{split}.json"

    with open(path, "r") as f:
        coco = json.load(f)

    # 🔴 THIS IS THE MISSING PART
    if "info" not in coco:
        coco["info"] = {
            "description": "Fire & Smoke dataset (YOLO → COCO converted)",
            "version": "1.0",
            "year": 2024
        }

    with open(path, "w") as f:
        json.dump(coco, f)

print("✅ COCO JSON files fixed (info field added)")


✅ COCO JSON files fixed (info field added)


In [9]:
dataset = DatasetCatalog.get("fire_train")
print(dataset[0])



Category ids in annotations are not in [1, #categories]! We'll apply a mapping for you.



{'file_name': '/kaggle/working/fire_subset/train/images/-2024-08-28-11_46_03_mov-0000_jpg.rf.6a34a77e4765e929c89ac8d5858b882c.jpg', 'height': 640, 'width': 640, 'image_id': 0, 'annotations': [{'iscrowd': 0, 'bbox': [8.421053, 87.21568599999999, 28.502024, 199.529412], 'category_id': 1, 'bbox_mode': <BoxMode.XYWH_ABS: 1>}, {'iscrowd': 0, 'bbox': [136.03238850000002, 153.0980395, 275.951417, 286.117647], 'category_id': 1, 'bbox_mode': <BoxMode.XYWH_ABS: 1>}, {'iscrowd': 0, 'bbox': [233.8461535, 139.9215685, 489.716599, 294.901961], 'category_id': 1, 'bbox_mode': <BoxMode.XYWH_ABS: 1>}, {'iscrowd': 0, 'bbox': [206.96356250000002, 123.6078425, 407.449393, 239.686275], 'category_id': 1, 'bbox_mode': <BoxMode.XYWH_ABS: 1>}]}


# Train

In [10]:
from detectron2.config import get_cfg
from detectron2 import model_zoo

cfg = get_cfg()
cfg.merge_from_file(
    model_zoo.get_config_file(
        "COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"
    )
)

cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
    "COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"
)


In [11]:
cfg.DATASETS.TRAIN = ("fire_train",)
cfg.DATASETS.TEST = ("fire_val",)

cfg.DATALOADER.NUM_WORKERS = 2

cfg.SOLVER.IMS_PER_BATCH = 4   # reduce to 2 if OOM
cfg.SOLVER.BASE_LR = 0.0025
cfg.SOLVER.MAX_ITER = 3000     # baseline experiment
cfg.SOLVER.STEPS = []

cfg.MODEL.ROI_HEADS.NUM_CLASSES = 2
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128

cfg.OUTPUT_DIR = "./output_faster_rcnn"
import os
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)


In [12]:
from detectron2.engine import DefaultTrainer
from detectron2.evaluation import COCOEvaluator
from detectron2.data import build_detection_test_loader

class TrainerWithValidation(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = cfg.OUTPUT_DIR
        return COCOEvaluator(
            dataset_name,
            cfg,
            False,
            output_folder
        )


In [13]:
from detectron2.engine import DefaultTrainer

trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()


[01/07 21:44:53 d2.engine.defaults]: Model:
GeneralizedRCNN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMaxPool()
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res

model_final_280758.pkl: 167MB [00:01, 86.5MB/s]                          
Skip loading parameter 'roi_heads.box_predictor.cls_score.weight' to the model due to incompatible shapes: (81, 1024) in the checkpoint but (3, 1024) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.cls_score.bias' to the model due to incompatible shapes: (81,) in the checkpoint but (3,) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.bbox_pred.weight' to the model due to incompatible shapes: (320, 1024) in the checkpoint but (8, 1024) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.bbox_pred.bias' to the model due to incompatible shapes: (320,) in the checkpoint but (8,) in the model! You might want to double check if this is expected.
Some model parameters or buffers are not found in the checkpoint:
roi_heads.box_pred

[01/07 21:44:55 d2.engine.train_loop]: Starting training from iteration 0


/usr/local/lib/python3.12/dist-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4322.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


[01/07 21:45:12 d2.utils.events]:  eta: 0:29:48  iter: 19  total_loss: 1.445  loss_cls: 0.9895  loss_box_reg: 0.3217  loss_rpn_cls: 0.1334  loss_rpn_loc: 0.02218    time: 0.5849  last_time: 0.4434  data_time: 0.0249  last_data_time: 0.0196   lr: 4.9952e-05  max_mem: 3062M


2026-01-07 21:45:18.004673: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767822318.544798      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767822318.677533      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767822319.950459      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767822319.950515      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767822319.950518      24 computation_placer.cc:177] computation placer alr

[01/07 21:45:52 d2.utils.events]:  eta: 0:31:35  iter: 39  total_loss: 0.947  loss_cls: 0.5055  loss_box_reg: 0.3171  loss_rpn_cls: 0.07845  loss_rpn_loc: 0.01417    time: 0.6130  last_time: 0.6498  data_time: 0.0149  last_data_time: 0.0102   lr: 9.9902e-05  max_mem: 3063M
[01/07 21:46:04 d2.utils.events]:  eta: 0:31:11  iter: 59  total_loss: 0.8609  loss_cls: 0.3575  loss_box_reg: 0.4083  loss_rpn_cls: 0.07747  loss_rpn_loc: 0.01703    time: 0.6135  last_time: 0.6239  data_time: 0.0178  last_data_time: 0.0362   lr: 0.00014985  max_mem: 3063M
[01/07 21:46:17 d2.utils.events]:  eta: 0:30:57  iter: 79  total_loss: 0.7808  loss_cls: 0.3286  loss_box_reg: 0.4111  loss_rpn_cls: 0.038  loss_rpn_loc: 0.01232    time: 0.6235  last_time: 0.5869  data_time: 0.0167  last_data_time: 0.0215   lr: 0.0001998  max_mem: 4688M
[01/07 21:46:30 d2.utils.events]:  eta: 0:30:57  iter: 99  total_loss: 0.929  loss_cls: 0.3769  loss_box_reg: 0.4882  loss_rpn_cls: 0.04967  loss_rpn_loc: 0.01755    time: 0.6314 

In [14]:
from detectron2.engine import DefaultPredictor
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader

cfg.DATASETS.TEST = ("fire_test",)
cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5

predictor = DefaultPredictor(cfg)

evaluator = COCOEvaluator(
    "fire_test",
    cfg,
    False,
    output_dir=cfg.OUTPUT_DIR
)

test_loader = build_detection_test_loader(cfg, "fire_test")
results = inference_on_dataset(
    predictor.model,
    test_loader,
    evaluator
)

results


[01/07 22:20:29 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from ./output_faster_rcnn/model_final.pth ...
WARNING [01/07 22:20:30 d2.evaluation.coco_evaluation]: COCO Evaluator instantiated using config, this is deprecated behavior. Please pass in explicit arguments instead.
WARNING [01/07 22:20:30 d2.data.datasets.coco]: 
Category ids in annotations are not in [1, #categories]! We'll apply a mapping for you.

[01/07 22:20:30 d2.data.datasets.coco]: Loaded 451 images in COCO format from /kaggle/working/fire_coco/test.json
[01/07 22:20:30 d2.data.build]: Distribution of instances among all 2 categories:
|  category  | #instances   |  category  | #instances   |
|:----------:|:-------------|:----------:|:-------------|
|    fire    | 413          |   smoke    | 133          |
|            |              |            |              |
|   total    | 546          |            |              |
[01/07 22:20:30 d2.data.dataset_mapper]: [DatasetMapper] Augmentations used

OrderedDict([('bbox',
              {'AP': 20.120760772925266,
               'AP50': 45.23912378073744,
               'AP75': 16.543731425572535,
               'APs': 12.517047884627297,
               'APm': 15.156256741849619,
               'APl': 23.637781578814078,
               'AP-fire': 28.382989044390573,
               'AP-smoke': 11.858532501459969})])